## Project Overview

This code demonstrates a structured approach to transforming raw interview data into a cleaner, more standardized dataset suitable for further analysis and integration into other systems. The data originates from interviews related to human trafficking cases and contains fields like interviewee demographics, experienced and witnessed abuses, border crossings, payment extortions, and known trafficker names. The ultimate goal is to unify and standardize the data, correct for inconsistencies, fill missing values where possible, and create a structured JSON output for each victim.

---

## Data Flow and Processing Steps

1. **Data Loading and Initial Setup**:  
   The process begins by reading in Excel and CSV files containing interview records and reference tables. The `filtered_data` DataFrame holds the main dataset.

2. **Column-Level Standardization**:  
   Various columns undergo different cleaning and standardization strategies to ensure the dataset is uniform and consistent across multiple entries. Key columns and their transformations include:
   
   - **"Interviewee number"**: Used as a unique identifier (`victim_id` in the JSON output).
   - **"Year of birth" and "Date of the interview"**:  
     Missing values are filled by grouping on "Interviewee number" and applying forward and backward fills. This ensures each interviewee has a consistent year of birth and interview date across all their rows.
   
   - **"Gender interviewee"**:  
     Inconsistent capitalization, trailing whitespace, and alternative forms ("male", "Male\n", "female") are standardized. A mapping dictionary is used to consolidate variations into a limited set of standard categories ("Male", "Female", "Group male/female", "Not Reported").
   
   - **"Nationality of the interviewee"**:  
     Various spellings, casing issues, and alternative forms of nationality descriptions are normalized. For instance, "Eriterian" and "Eritrean" are both mapped to "Eritrea". The code references a CSV from a previous project by Kai to guide these mappings, ensuring that all nationalities are represented by a standard country name or "Unknown" if unspecified.
   
   - **"Human traffickers/ Chief of places" (extracted_known_traffickers)**:  
     Trafficker names appear in numerous spellings, formats, and with additional descriptive text. Using a multi-step mapping process—starting with raw-to-processed conversions, then applying standardized mappings—entries are reduced to a set of canonical trafficker names. This leverages a CSV provided by Kai, who shared an authoritative reference for known trafficker names. Complex entries like "Not specified, Kidane" or "Chadian soldiers, Kidane" are ultimately standardized to "Kidane," and variants of "Abdela Muhamad And Muhamad And Abdela" are mapped to "Abdela and Muhamad."
   
   - **"Payment"** (cleaned_payment_amount, cleaned_payment_currency):  
     Currency symbols, textual descriptions, and amounts are extracted and converted into standardized forms. Terms like "dollars," "usd," or "$" become "usd," while amounts are parsed out of text. Non-numeric or ambiguous cases are labeled as "unknown" or "not specified."
   
   - **"Abuses experienced", "Abuses seen"**:
     Using keyword-based classification, abuse descriptions are categorized into "Physical Abuse," "Sexual Abuse," "Psychological Abuse," "Torture," "Neglect," or "Other Abuse" depending on the presence of certain keywords. Null or unspecified values are handled by returning "no" or "none," ensuring a consistent format.
   
   - **"sexual violence experienced/witnessed" and "deaths witnessed" columns**:  
     These columns are similarly processed using keyword searches and logic to return binary indicators ("yes", "no", "no data") and specific types ("rape", "none", "not specified"). Missing data or uncategorizable entries result in "no data" or "unknown" values.
   
   - **"Locations crossed (Place from)" and "Locations crossed (Place to)"**:  
     The route and place information is standardized against a location mapping dictionary. This dictionary was refined using previous group projects, ensuring consistent geographical naming conventions. Various alternate spellings, typos, and language forms are normalized to a single known location (e.g., "Lybia" and "Libya" both become "Libya"). Unknown or ambiguous places are set to "Unknown."

3. **Handling Missing and Null Values**:  
   The script uses forward-fill and backward-fill operations (`ffill()` and `bfill()`) within groups based on "Interviewee number" to handle missing demographic data such as year of birth or date of interview. Other columns are assigned "not specified" or "Unknown" when no clear mapping is possible. The `.fillna()` method ensures that if a particular value doesn't match a given mapping, it falls back to the original value or a known default, preserving data integrity.

4. **Calculating Derived Fields**:  
   With year of birth and interview date standardized, an `Age` field is computed. The code attempts multiple date formats and falls back to "not specified" if the calculation cannot be completed due to parsing issues or missing data.

5. **JSON Creation**:  
   After the data is fully standardized, a JSON structure is assembled for each interviewee row. A `create_json` function organizes data into a nested dictionary:
   - **victim_id, age, gender, nationality**: Key victim attributes.
   - **trafficker_name**: A standardized trafficker name from the processed column.
   - **crimes**: A list of dictionaries capturing different facets of sexual violence experienced or witnessed, deaths witnessed, and abuses.  
   - **borders_crossed**: A list holding "from" and "to" locations of border crossings.
   - **money_extorted_amount and money_extorted_currency**: Standardized financial extortion details.

   The list of victim dictionaries is then serialized into JSON using `json.dumps()` and saved to a file (`victims.json`).

---

## Final Output

By following these steps, the raw, inconsistent interview data is transformed into a clean and uniform dataset. The final JSON output provides a standardized, machine-readable representation of each victim's information, including personal attributes, encountered traffickers, types of abuses, border crossings, and extortion details. This standardized format supports easier integration with analytical tools, reporting systems, and further research on trafficking patterns and victim experiences.

In [1]:
import pandas as pd

"""
This script demonstrates how to load and preprocess interview data from an Excel file.
It performs the following operations:
1. Reads the specified Excel file and extracts data from the first sheet.
2. Selects a subset of relevant columns for further analysis.
3. Displays the shape of the filtered DataFrame to confirm successful processing.

Prerequisites:
- The Excel file (data.xlsx) must be present in the same directory as the script, or
  `file_path` should point to the correct location.
- The first sheet must exist and contain headers.
- The specified columns in `selected_columns` must match the headers in the sheet.

Dependencies:
- pandas (imported as pd)

Returns:
- None (prints out the shape of the resulting filtered DataFrame)
"""

# Load the Excel file
file_path = 'data.xlsx'
excel_data = pd.ExcelFile(file_path)

# Parse the first sheet (whatever it's named)
sheet_name = excel_data.sheet_names[0]
print(f"Reading sheet: {sheet_name}")
interview_data = excel_data.parse(sheet_name)

# Selecting only the columns specified
selected_columns = [
    'Interviewee number','Date of the interview','Year of birth','Interview ranking number', 'smugglers/traffickers',
    'human rights abuses mentioned', 'sexual violence experienced',
    'Sexual violence witnessed', 'type of sexual violences: rape, pregnancy, human trafficking',
    'Deaths witnessed', 'Human traffickers/ Chief of places', 'Abuses experienced',
    'Abuses seen', 'borders crossed', 'Payment','Nationality of the interviewee','Gender interviewee'
]

# Filter the data to include only these columns
filtered_data = interview_data[selected_columns]

# Display the filtered data
filtered_data.shape

Reading sheet: Sheet1


(100, 17)

In [2]:
# Check the unique values present in the 'sexual violence experienced' column.
# This will help identify the different categories or descriptors used for sexual violence.
# The results obtained from this analysis can be further used as input to a locally hosted
# language model (via LMStudio) to generate standardized dictionaries or mappings.
# These dictionaries can then be used to process and normalize the data for analysis,
# ensuring consistent terminology and categorization.
# Checking the unique values in the 'sexual violence experienced' column
unique_values = filtered_data['sexual violence experienced'].unique()
unique_values


array(['Yes', 'No'], dtype=object)

In [3]:
'''
Extract origin and destination locations from the 'borders crossed' column.
The 'borders crossed' field is assumed to contain strings that represent two locations 
separated by a forward slash ('/'), where the portion before the slash is the origin and 
the portion after the slash is the destination.

By splitting this column into two separate fields, 'Locations crossed (Place from)' 
and 'Locations crossed (Place to)', further analysis can be conducted on these distinct 
elements (e.g., identifying migration paths, analyzing frequency of certain borders crossed, etc.).

If there is only a single location and no slash present, it means we don't have a clear 
"to" location, resulting in 'Locations crossed (Place to)' being null (NaN).
'''
filtered_data[['Locations crossed (Place from)', 'Locations crossed (Place to)']] = filtered_data['borders crossed'].str.split('/', n=1, expand=True)
'''
Create a new column 'border crossed (undefined)' to handle cases where only one location 
was provided. If 'Locations crossed (Place to)' is missing, we assume the border crossing 
is not well-defined and store the original single-location value in 'border crossed (undefined)'.

This step ensures that all border-related information is properly categorized:
- Rows with a well-defined origin and destination are split into two columns.
- Rows with a single location are flagged in the 'border crossed (undefined)' column, 
  allowing separate handling or cleaning in subsequent processing stages.
'''
filtered_data['border crossed (undefined)'] = filtered_data.apply(
    lambda row: row['borders crossed'] if pd.isnull(row['Locations crossed (Place to)']) else None, axis=1
)

/tmp/ipykernel_4785/2108927107.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['Locations crossed (Place from)', 'Locations crossed (Place to)']] = filtered_data['borders crossed'].str.split('/', n=1, expand=True)
/tmp/ipykernel_4785/2108927107.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['Locations crossed (Place from)', 'Locations crossed (Place to)']] = filtered_data['borders crossed'].str.split('/', n=1, expand=True)
/tmp/ipykernel_4785/2108927107.py:25: Settin

In [4]:
import pandas as pd

'''
This code snippet refines classification of sexual violence experiences mentioned in interview data.

Steps:
1. Define a function `classify_sexual_violence(row)` that takes a string describing sexual violence experiences:
   - If the value is missing, it classifies as "no data".
   - If terms like "none" or "no" appear, it classifies as "no" sexual violence.
   - If explicit terms like "rape", "sexual assault", or "sexual violence" appear, it classifies as "yes" and categorizes as "rape".
   - If the description implies sexual violence but not explicitly as "rape", it classifies as "yes" and "not specified".
   - Otherwise, it falls back to "yes" and "not specified" for ambiguous cases.
   
2. The classification results are stored in two new columns:
   - 'sexual_violence_experienced_binary': Indicates yes/no/no data classification.
   - 'sexual_violence_experienced_type': Specifies the nature of sexual violence (e.g., 'rape', 'none', 'not specified').

3. After applying the classification function, the resulting DataFrame columns are printed to verify outputs.

Note:
This classification system can be further refined by using dictionaries generated by a locally hosted language model (e.g., LMStudio) to standardize and normalize the terminology. Such dictionaries can help ensure consistent categories across various input descriptions.
'''

'''
The classify_sexual_violence function takes a single description (row) of sexual violence,
checks for known keywords and phrases, and returns a tuple (binary_classification, type_classification).

Return values:
- ('no data', 'no data'): For missing or null entries.
- ('no', 'none'): For descriptions explicitly indicating no sexual violence.
- ('yes', 'rape'): For explicit mentions of sexual violence like "rape" or "sexual assault".
- ('yes', 'not specified'): For descriptions indicating sexual violence but not explicitly categorized as "rape".
'''

# Load the CSV fi
def classify_sexual_violence(row):
    # Checking for missing data or unspecified cases
    if pd.isnull(row):
        return 'no data', 'no data'
    
    description = str(row).lower()
    
    # Checking for cases indicating "no" or "none" mentioned
    if any(phrase in description for phrase in ['none mentioned', 'none', 'no', 'not mentioned', 'not specified', 'not reported']):
        return 'no', 'none'
    
    # Specific terms indicating "yes" with 'rape' or similar sexual violence
    if any(keyword in description for keyword in ['rape', 'sexual assault', 'sexual violence', 'exploitation']):
        return 'yes', 'rape'
    
    # Specific phrases indicating sexual violence but without clear categorization as 'rape'
    if any(keyword in description for keyword in [
        'tried to abuse', 'threatened to be raped', 'taking us out', 
        'taking us in shifts', 'exploitation', 'sexual assaults'
    ]):
        return 'yes', 'not specified'
    
    # Handling cases indicating difficult circumstances without clear mention of sexual violence
    if any(keyword in description for keyword in ['food', 'toilet', 'water', 'sickness', 'beat', 'violence', 'problems']):
        return 'no', 'none'
    
    # Catch-all for ambiguous cases suggesting yes without specific categorization
    return 'yes', 'not specified'

# Apply the refined classification function to create two new columns
filtered_data[['sexual_violence_experienced_binary', 'sexual_violence_experienced_type']] = filtered_data['sexual violence experienced'].apply(lambda x: pd.Series(classify_sexual_violence(x)))

# Display the updated DataFrame to verify the output (optional)
print(filtered_data[['sexual violence experienced', 'sexual_violence_experienced_binary', 'sexual_violence_experienced_type']].head(20))


   sexual violence experienced sexual_violence_experienced_binary  \
0                          Yes                                yes   
1                           No                                 no   
2                           No                                 no   
3                          Yes                                yes   
4                           No                                 no   
5                          Yes                                yes   
6                          Yes                                yes   
7                          Yes                                yes   
8                          Yes                                yes   
9                           No                                 no   
10                          No                                 no   
11                          No                                 no   
12                         Yes                                yes   
13                          No    

/tmp/ipykernel_4785/2765163771.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['sexual_violence_experienced_binary', 'sexual_violence_experienced_type']] = filtered_data['sexual violence experienced'].apply(lambda x: pd.Series(classify_sexual_violence(x)))
/tmp/ipykernel_4785/2765163771.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['sexual_violence_experienced_binary', 'sexual_violence_experienced_type']] = filtered_data['sexual violence experienced'].apply(lambda x

In [5]:
'''
This code defines a function `classify_sexual_violence_witnessed` to categorize the nature of sexual
violence witnessed as described in interview data. The function takes a single row value 
from the 'Sexual violence witnessed' column and classifies it into two parts:
1. A binary indicator ('yes', 'no', or 'no data').
2. A type classifier ('none', 'rape', 'not specified').

Classification logic:
- If the entry is missing, returns ('no data', 'no data').
- If terms like "none" or "no" appear, returns ('no', 'none').
- If explicit terms like "rape", "sexual violence", "prostitution", "sex slave", or "exploitation"
  appear, returns ('yes', 'rape').
- If there are clues suggesting witnessed violence but not explicitly categorized as "rape"
  (e.g., "taking women out", "heard", "threatened", "sexual assaults", "abuse"), returns 
  ('yes', 'not specified').
- Ambiguous cases default to ('yes', 'not specified').

The classification results are then applied to the 'Sexual violence witnessed' column of 
`filtered_data`, producing two new columns:
- 'sexual_violence_witnessed_binary': A yes/no/no data indicator.
- 'sexual_violence_witnessed_type': The type of sexual violence witnessed (e.g., 'rape', 'none', 'not specified').

These standardized categories assist in downstream analysis, ensuring consistent and 
comparable data points across various interviews.
'''

def classify_sexual_violence_witnessed(row):
    # Handle missing or unspecified data
    if pd.isnull(row):
        return 'no data', 'no data'
    
    description = str(row).lower()
    
    # Explicitly stated as no violence witnessed
    if any(phrase in description for phrase in ['none mentioned', 'none', 'no', 'not mentioned', 'not specified', 'not reported']):
        return 'no', 'none'
    
    # Cases with clear indication of witnessing rape or related sexual violence
    if any(keyword in description for keyword in ['rape', 'sexual violence', 'prostitution', 'sex slave', 'exploitation']):
        return 'yes', 'rape'
    
    # Cases implying some witnessed violence but not specified
    if any(keyword in description for keyword in [
        'taking women out', 'heard', 'threatened', 'sexual assaults', 'abuse'
    ]):
        return 'yes', 'not specified'
    
    # Default classification for ambiguous cases suggesting yes
    return 'yes', 'not specified'

'''
Apply the classification function to the 'Sexual violence witnessed' column of the DataFrame.
The results are stored in two new columns:
- 'sexual_violence_witnessed_binary'
- 'sexual_violence_witnessed_type'

These columns allow standardized analysis of witnessed sexual violence cases.
'''

# Apply the classification function to create two new columns for sexual violence witnessed
filtered_data[['sexual_violence_witnessed_binary', 'sexual_violence_witnessed_type']] = filtered_data['Sexual violence witnessed'].apply(lambda x: pd.Series(classify_sexual_violence_witnessed(x)))

/tmp/ipykernel_4785/461466739.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['sexual_violence_witnessed_binary', 'sexual_violence_witnessed_type']] = filtered_data['Sexual violence witnessed'].apply(lambda x: pd.Series(classify_sexual_violence_witnessed(x)))
/tmp/ipykernel_4785/461466739.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['sexual_violence_witnessed_binary', 'sexual_violence_witnessed_type']] = filtered_data['Sexual violence witnessed'].apply(lambda x: pd

In [6]:
# Define a function to classify deaths witnessed based on the observed unique values
def classify_deaths_witnessed(row):
    # Handle missing or unspecified data
    if pd.isnull(row):
        return 'no data', 'no data'
    
    description = str(row).lower()
    
    # Explicitly stated as no deaths witnessed
    if any(phrase in description for phrase in ['none mentioned', 'none', 'no', 'not mentioned', 'not applicable', 'not reported']):
        return 'no', 'none'
    
    # Cases with clear indication of witnessed deaths due to violence
    if any(keyword in description for keyword in ['killed', 'shot', 'murdered', 'violence', 'war']):
        return 'yes', 'violence'
    
    # Cases indicating deaths due to sickness, hunger, or lack of medical care
    if any(keyword in description for keyword in ['sickness', 'disease', 'hunger', 'starved', 'tb', 'illness', 'medicine', 'diarrhoea']):
        return 'yes', 'sickness'
    
    # Cases indicating deaths due to accidents, such as car accidents or drowning
    if any(keyword in description for keyword in ['accident', 'drown', 'fell', 'car', 'crash', 'thrown', 'vehicle']):
        return 'yes', 'accident'
    
    # Default classification for ambiguous cases
    return 'yes', 'not specified'

# Apply the classification function to create two new columns for deaths witnessed
filtered_data[['deaths_witnessed_binary', 'deaths_witnessed_type']] = filtered_data['Deaths witnessed'].apply(lambda x: pd.Series(classify_deaths_witnessed(x)))

# Display the updated DataFrame to verify the output
filtered_data[['Deaths witnessed', 'deaths_witnessed_binary', 'deaths_witnessed_type']].head(20)


/tmp/ipykernel_4785/3470922553.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['deaths_witnessed_binary', 'deaths_witnessed_type']] = filtered_data['Deaths witnessed'].apply(lambda x: pd.Series(classify_deaths_witnessed(x)))
/tmp/ipykernel_4785/3470922553.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['deaths_witnessed_binary', 'deaths_witnessed_type']] = filtered_data['Deaths witnessed'].apply(lambda x: pd.Series(classify_deaths_witnessed(x)))


,Deaths witnessed,deaths_witnessed_binary,deaths_witnessed_type
0,0,yes,not specified
1,2,yes,not specified
2,0,yes,not specified
3,3,yes,not specified
4,1,yes,not specified
5,2,yes,not specified
6,4,yes,not specified
7,4,yes,not specified
8,1,yes,not specified
9,0,yes,not specified


In [7]:
import pandas as pd
import re
'''
This code processes and standardizes payment-related information within the dataset. It uses a custom function 
`classify_payment` to:

1. Identify the payment amount from textual descriptions using pattern matching.
2. Map various currency symbols and references to standardized currency codes (e.g., '$' -> 'usd', '€' -> 'eur').
3. Handle edge cases such as:
   - Missing or null values (classified as "no data").
   - Statements indicating no payment or unspecified amounts ("no payment", "not specified").
   - Entries mentioning work or family-related remittances, which are treated differently.

By applying the classification function to the 'Payment' column of `filtered_data`, the code produces two new columns:
- 'cleaned_payment_amount': The extracted or derived amount (or "no payment"/"not specified").
- 'cleaned_payment_currency': The standardized currency code or "unknown currency".

This standardized and structured payment data facilitates consistent analysis and comparisons across different records.
'''

# Define a function to classify and clean payment data, mapping symbols to currency names
def classify_payment(entry):
    '''
    Classifies payment information from a given text entry.
    
    Parameters
    ----------
    entry : str or NaN
        The raw textual description of payment, which may contain
        currency symbols, amounts, or other notes.
    
    Returns
    -------
    pd.Series
        A two-element Series:
        [0] : str
            The standardized amount (e.g., a numeric value as a string, 
            "no data", "no payment", "not specified", or "unknown").
        [1] : str
            The standardized currency code (e.g., "usd", "eur", "gbp", 
            or "unknown currency").
    '''
    if pd.isnull(entry):
        return pd.Series(["no data", "no data"])  # Indicate missing data
    
    # Convert to lowercase for uniform processing
    description = str(entry).lower()

    # Regular expression for amounts and standardized mapping for currency symbols/names
    amount_pattern = r'(\d+(?:[\s,]?\d+)*(?:\.\d+)?)'
    currency_map = {
        r'\$': 'usd', r'usd': 'usd', r'dollers?': 'usd', r'dollars?': 'usd', 
        r'euros?': 'eur', r'€': 'eur', r'pounds?': 'gbp', r'£': 'gbp',
        r'dinars?': 'dinar', r'birr': 'birr', r'cfa': 'cfa', r'fcfa': 'cfa', 
        r'dalasi': 'dalasi', r'naira': 'naira'
    }
    
    # Find the amount in the text if present
    amount_match = re.search(amount_pattern, description.replace(",", ""))
    amount = amount_match.group(0) if amount_match else "unknown"

    # Apply currency mapping by searching for patterns in description text
    currency = "unknown currency"
    for pattern, standard_currency in currency_map.items():
        if re.search(pattern, description):
            currency = standard_currency
            break

    # Handle cases with phrases indicating unspecified or varying payments
    if "not specified" in description or "amount not mentioned" in description or "unspecified" in description:
        amount = "not specified"
    elif "worked" in description or "work" in description or "job" in description:
        return pd.Series(["no payment", "not specified"])
    elif "sent money" in description or "family" in description:
        return pd.Series(["not specified", "unknown currency"])

    # Return the classified amount and standardized currency name
    return pd.Series([amount, currency])

'''
Apply the classify_payment function to the 'Payment' column to create standardized 
'cleaned_payment_amount' and 'cleaned_payment_currency' columns.
After processing, the data is displayed to verify that currency symbols and references 
have been correctly identified and standardized.
'''

# Apply the function to classify amounts and replace symbols with currency names
filtered_data[['cleaned_payment_amount', 'cleaned_payment_currency']] = filtered_data['Payment'].apply(classify_payment)

# Display results to verify currency name replacements
filtered_data[['Payment', 'cleaned_payment_amount', 'cleaned_payment_currency']].dropna().tail(20)


/tmp/ipykernel_4785/2444396044.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['cleaned_payment_amount', 'cleaned_payment_currency']] = filtered_data['Payment'].apply(classify_payment)
/tmp/ipykernel_4785/2444396044.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['cleaned_payment_amount', 'cleaned_payment_currency']] = filtered_data['Payment'].apply(classify_payment)


,Payment,cleaned_payment_amount,cleaned_payment_currency
80,$3388,3388,usd
81,$1756,1756,usd
82,$4155,4155,usd
83,$3461,3461,usd
84,$1104,1104,usd
85,$4750,4750,usd
86,$2668,2668,usd
87,$3904,3904,usd
88,$1293,1293,usd
89,$2472,2472,usd


In [8]:
import pandas as pd


'''
This code maps mentions of human traffickers in the interview dataset to a known set of primary and alternative names 
derived from an Excel file ('data.xlsx'). This initial mapping ensures that references to the same individual, 
despite variations in spelling or nicknames, are linked to a single, canonical name.

Process:
1. Load the known trafficker information (including alternative names) from the trafficker sheet.
2. Create a comprehensive set of names (primary and alternative) and convert them to lowercase for case-insensitive matching.
3. Define a function that identifies any known trafficker names mentioned in a given interview entry.
   - If matches are found, they are returned in a comma-separated list.
   - If no match is found or the entry is missing, "unknown" is returned.
4. Apply this function to the 'Human traffickers/ Chief of places' column to produce an 'extracted_known_traffickers' column.

This initial mapping step ensures consistency in identifying individuals across multiple interviews. 
Later, these matched names will be further standardized using another CSV file provided by the mentor, 
which contains more authoritative or consolidated naming conventions. 
This two-step process (initial mapping followed by standardization) helps ensure data quality and 
improves the reliability of subsequent analyses.
'''

import pandas as pd

# Load the Excel file and read the traffickers sheet
excel_path = 'data.xlsx'
excel_data = pd.ExcelFile(excel_path)

# Try to find the traffickers sheet - it might be named differently
trafficker_sheet = None
for sheet in excel_data.sheet_names:
    if 'trafficker' in sheet.lower() or 'MASTER SHEET-2' in sheet:
        trafficker_sheet = sheet
        break

if trafficker_sheet:
    print(f"Found trafficker sheet: {trafficker_sheet}")
    trafficker_data = excel_data.parse(trafficker_sheet)
    
    # Map column names - the sheet might use different names
    # Expected: 'Name', 'Alternative names'
    # Actual: 'Trafficker Name', 'Alt Name'
    if 'Trafficker Name' in trafficker_data.columns:
        trafficker_data = trafficker_data.rename(columns={'Trafficker Name': 'Name', 'Alt Name': 'Alternative names'})
    
    # Extract primary trafficker names (in lowercase)
    trafficker_names = trafficker_data['Name'].dropna().str.lower().tolist()
    
    # Extract alternative names, handle multiple comma-separated entries, and lowercase them
    alternative_names = trafficker_data['Alternative names'].dropna().str.lower().str.split(',').tolist()
    alternative_names_flat = [name.strip() for sublist in alternative_names for name in sublist]
    
    # Combine primary and alternative names into a single set to avoid duplicates
    all_trafficker_names = set(trafficker_names + alternative_names_flat)
else:
    print("Warning: No trafficker sheet found, using empty list")
    all_trafficker_names = set()

def extract_traffickers_with_alternatives(entry):
    '''
    Identifies any known traffickers mentioned in a given text entry.
    Returns a comma-separated string of matched trafficker names or "unknown" if none is found.
    '''
    if pd.isnull(entry):
        return "unknown"
    
    if not all_trafficker_names:
        return "unknown"
    
    description = str(entry).lower()
    matched_traffickers = list({name.capitalize() for name in all_trafficker_names if name in description})
    return ", ".join(matched_traffickers) if matched_traffickers else "unknown"

# Apply the extraction function to map the interview data to known trafficker names
filtered_data['extracted_known_traffickers'] = filtered_data['Human traffickers/ Chief of places'].apply(extract_traffickers_with_alternatives)

# Display a sample of the results to verify that alternative names have been included
filtered_data[['Human traffickers/ Chief of places', 'extracted_known_traffickers']].head(20)


/tmp/ipykernel_4785/3339234939.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['extracted_known_traffickers'] = filtered_data['Human traffickers/ Chief of places'].apply(extract_traffickers_with_alternatives)


,Human traffickers/ Chief of places,extracted_known_traffickers
0,Yes,unknown
1,Yes,unknown
2,Yes,unknown
3,No,unknown
4,Yes,unknown
5,Yes,unknown
6,Yes,unknown
7,No,unknown
8,Yes,unknown
9,No,unknown


In [9]:
'''
To better understand the range of abuses mentioned in the interviews, we first retrieve 
and inspect the unique values present in the 'Abuses experienced' column. This step 
provides insight into the variety and wording of responses, which may include explicit 
terms, shorthand references, or unclear phrases.

By extracting these unique values, we lay the groundwork for creating standardized 
dictionaries or categorizations. These unique entries will be used as input to a 
locally hosted language model (LLM), which will help generate structured mappings 
and classification rules. Such standardized references can then be applied back 
to the dataset, enabling consistent analysis and comparison across different interviews.
'''
unique_values_abuses = filtered_data['Abuses experienced'].unique()
unique_values_abuses


array(['Beating', 'Torture', nan, 'Starvation', 'Forced labor'],
      dtype=object)

In [10]:
import pandas as pd
import numpy as np

'''
This code segment classifies the types of abuses mentioned in the 'Abuses experienced' column of the dataset 
into standardized categories. By scanning for keywords associated with different forms of abuse (physical, 
sexual, psychological), we can more easily analyze and compare responses across interviews.

Process:
1. Check for missing or null values. If found, classify as "no" and "none".
2. Define sets of terms representing various abuse categories:
   - Physical (e.g., "beaten", "tortured", "violence")
   - Sexual (e.g., "rape", "sexual", "assault")
   - Psychological (e.g., "threat", "humiliate", "fear")
3. For each description, determine which categories are present based on these keywords.
   - If no category is identified, label it as "General Abuse (Unspecified)".
4. The result is stored in two new columns:
   - 'abuse_type' (a comma-separated string of matched abuse categories)
   - 'abuse_labels' (a list of matched categories for more flexible analysis)

This standardized classification facilitates more systematic data exploration, enabling 
downstream analyses such as frequency counts, correlation with other variables, and 
more nuanced reporting of the types of abuse experienced by interviewees.
'''


# Define the classification function for "Abuses experienced"
def classify_abuse(description):
    if pd.isnull(description) or "none" in description.lower():
        return pd.Series(["no", "none"])

    physical_terms = ["beaten", "tortured", "burned", "injured", "killed", "shot", "violence", "electricity", "stick", "hit", "broken", "beating", "hurt", "abuse", "food", "hunger", "drink"]
    sexual_terms = ["rape", "sexual", "assault", "abuse", "molesting", "menstruation"]
    psychological_terms = ["threat", "humiliate", "mental", "psychological", "scared", "fear", "cry", "shame", "pressure"]
    
    physical = any(term in description.lower() for term in physical_terms)
    sexual = any(term in description.lower() for term in sexual_terms)
    psychological = any(term in description.lower() for term in psychological_terms)

    labels = []
    if physical:
        labels.append("Physical Abuse")
    if sexual:
        labels.append("Sexual Abuse")
    if psychological:
        labels.append("Psychological Abuse")
    if not labels:  # If no specific category matched, label as General
        labels.append("General Abuse (Unspecified)")
    
    return pd.Series([", ".join(labels), labels])

# Apply the classification function to the column
filtered_data[['abuse_type', 'abuse_labels']] = filtered_data['Abuses experienced'].apply(classify_abuse)

# Display first 10 rows for verification
display_output = filtered_data[['Abuses experienced', 'abuse_type', 'abuse_labels']].head(10)
print(display_output)


  Abuses experienced                   abuse_type  \
0            Beating               Physical Abuse   
1            Beating               Physical Abuse   
2            Torture  General Abuse (Unspecified)   
3            Torture  General Abuse (Unspecified)   
4                NaN                           no   
5            Beating               Physical Abuse   
6                NaN                           no   
7         Starvation  General Abuse (Unspecified)   
8            Beating               Physical Abuse   
9            Torture  General Abuse (Unspecified)   

                    abuse_labels  
0               [Physical Abuse]  
1               [Physical Abuse]  
2  [General Abuse (Unspecified)]  
3  [General Abuse (Unspecified)]  
4                           none  
5               [Physical Abuse]  
6                           none  
7  [General Abuse (Unspecified)]  
8               [Physical Abuse]  
9  [General Abuse (Unspecified)]  


/tmp/ipykernel_4785/1945619562.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data[['abuse_type', 'abuse_labels']] = filtered_data['Abuses experienced'].apply(classify_abuse)


In [11]:
'''
This snippet retrieves all the unique values from the 'Abuses seen' column to understand 
the various ways interviewees describe witnessed abuses. By identifying these unique 
entries, we gain insight into the range and complexity of the terminology used.

These unique values will later be provided to a locally hosted language model (LLM) 
to assist in generating structured mappings and standardized categories. Such 
standardization ensures consistent analysis and clearer understanding of how 
witnessed abuses are described across the entire dataset.'''

# Checking unique values in the 'Abuses experienced' column to understand the data for standardization
unique_values_abuses = filtered_data['Abuses seen'].unique()
unique_values_abuses


array(['Killing', nan, 'Beating', 'Torture'], dtype=object)

In [12]:
'''
This code snippet classifies the abuses seen by interviewees into specific categories 
based on keywords present in the "Abuses seen" column. By scanning each entry for 
indicators of physical abuse, sexual abuse, torture, neglect, psychological abuse, 
or the absence of abuse (none), the function standardizes a wide variety of textual 
descriptions into a set of predefined categories. If the keywords do not match any 
of the known categories, the entry is classified as "Other Abuse".

Process:
1. Convert the input value to lowercase to ensure case-insensitive keyword matching.
2. Check for category-specific keywords:
   - Physical Abuse: Terms like "beat", "hit", "shooting", etc.
   - Sexual Abuse: Terms like "rape", "sexual", "molest", "assault"
   - Torture: Terms like "torture", "electric", "shock", "burn"
   - Neglect: Terms like "food", "water", "hunger"
   - Psychological Abuse: Terms like "fear", "threat", "mental", "psychological"
   - None: Terms indicating the absence of abuse ("none mentioned", "no")
3. If no category matches, classify as "Other Abuse".

This standardized classification makes it easier to analyze the types of abuses 
witness'''


# Define the function to classify based on keywords
def classify_abuses_seen(value):
    # Convert value to lowercase for uniform comparison
    value = str(value).lower() if isinstance(value, str) else ""
    
    # Physical Abuse
    if any(keyword in value for keyword in ['beat', 'beating', 'hit', 'shooting', 'scare', 'tied', 'chain', 'physical', 'fighting', 'injury']):
        return 'Physical Abuse'
    
    # Sexual Abuse
    elif any(keyword in value for keyword in ['rape', 'sexual', 'molest', 'assault']):
        return 'Sexual Abuse'
    
    # Torture
    elif any(keyword in value for keyword in ['torture', 'electric', 'shock', 'burn', 'suffocate', 'whip', 'drill', 'stick', 'plastic fire']):
        return 'Torture'
    
    # Neglect
    elif any(keyword in value for keyword in ['food', 'water', 'hunger', 'starving', 'neglect', 'no respect', 'unhygienic']):
        return 'Neglect'
    
    # Psychological Abuse
    elif any(keyword in value for keyword in ['fear', 'scare', 'threat', 'intimidate', 'insult', 'humiliate', 'mental', 'psychological']):
        return 'Psychological Abuse'
    
    # None
    elif any(keyword in value for keyword in ['none mentioned', 'not specified', 'no']):
        return 'None'
    
    # Other Abuse
    else:
        return 'Other Abuse'

# Apply the classification function to the "Abuses seen" column
filtered_data['abuse_seen_type'] = filtered_data['Abuses seen'].apply(classify_abuses_seen)



In [13]:
# Drop the specified columns from the DataFrame
filtered_data.drop(columns=['smugglers/traffickers', 'human rights abuses mentioned',
       'sexual violence experienced', 'Sexual violence witnessed',
       'type of sexual violences: rape, pregnancy, human trafficking',
       'Deaths witnessed', 'Human traffickers/ Chief of places',
       'Abuses experienced', 'Abuses seen', 'borders crossed', 'Payment',
], inplace=True)




In [14]:
'''
In some cases, data for certain attributes (such as nationality or gender) may be missing 
for specific interviews even though the same individual (Interviewee number) is discussed 
across multiple rows. To ensure consistency, we can propagate existing values for these 
attributes forward and backward within each group of interviews belonging to the same 
individual.

Process:
1. Group the data by 'Interviewee number'.
2. For each group, use forward fill (ffill) and backward fill (bfill) on the 'Nationality of the interviewee' 
   and 'Gender interviewee' columns.
   - Forward fill (ffill) replaces missing values with the last known value from above.
   - Backward fill (bfill) replaces missing values with the next known value from below.
   
By applying both ffill and bfill, we ensure that each interviewee's nationality and gender 
are consistently assigned to all their associated rows, improving data completeness and 
coherence for subsequent analysis.
'''

filtered_data['Nationality of the interviewee'] = filtered_data.groupby('Interviewee number')['Nationality of the interviewee'].transform(lambda x: x.ffill().bfill())
filtered_data['Gender interviewee'] = filtered_data.groupby('Interviewee number')['Gender interviewee'].transform(lambda x: x.ffill().bfill())


In [15]:
'''
This code standardizes the gender values found in the "Gender interviewee" column. 
Because interview data may contain inconsistent or messy gender labels, we define a 
mapping dictionary that normalizes these values into a consistent set of categories.

Steps:
1. Create a `gender_mapping` dictionary that maps various forms of male/female and 
   other reported values to standardized strings (e.g., "Male", "Female", "Not Reported").
   
2. Use the `str.strip()` method to remove any trailing whitespace (e.g., "Male\n").
3. Apply the `map()` function with `fillna()` to ensure that unmapped values remain 
   unchanged (if any).
   
After this process, all gender values are more uniform, enabling more reliable 
analysis and easier categorization.
'''


gender_mapping = {
    "Male": "Male",
    "male": "Male",
    "Male\n": "Male",
    "Female": "Female",
    "female": "Female",
    "Group male/female": "Group male/female",
    "not reported": "Not Reported",
}

# Apply the mapping to the 'Gender interviewee' column
filtered_data["Gender interviewee"] = filtered_data["Gender interviewee"].str.strip().map(gender_mapping).fillna(filtered_data["Gender interviewee"])

# Check the standardized values
print(filtered_data["Gender interviewee"].value_counts())

Gender interviewee
Male      41
Female    30
Other     29
Name: count, dtype: int64


In [16]:
'''
This code snippet performs an initial standardization of nationality values by correcting 
inconsistent casing, removing extra whitespace, and mapping various forms of country names 
to a consistent set of standardized terms. The objective is not a complete 
standardization—just a preliminary cleanup—ensuring that commonly encountered variants 
(e.g., "Eriterian", "\ncameroonian") are normalized to a recognized form (e.g., "Eritrean", "Cameroonian").

Process:
1. Define a mapping dictionary (`nationality_mapping`) that maps different variations of nationality 
   strings to a standardized version (e.g., lowercase, alternate spellings).
2. Strip extra whitespace and apply the mapping using `str.strip()` and `.map()`. 
   Any values not found in the dictionary remain unchanged due to the `fillna()` fallback.

After this step, nationality values are more uniform, making further standardization and analysis easier.
'''



# Define a mapping dictionary for standardizing nationalities
nationality_mapping = {
    "Eritrean": "Eritrean",
    "eritrean": "Eritrean",
    "Guinean": "Guinean",
    "guinean": "Guinean",
    "Senegalese": "Senegalese",
    "senegalese": "Senegalese",
    "somali": "Somali",
    "Somali": "Somali",
    "Bissau-Guinean": "Bissau-Guinean",
    "Nigerian": "Nigerian",
    "Somalian": "Somalian",
    "Cameroonian": "Cameroonian",
    "cameroonian": "Cameroonian",
    "\ncameroonian": "Cameroonian",
    "Ivorian": "Ivorian",
    "Tunisian": "Tunisian",
    "Eriterian": "Eritrean",
    "from Cameroon (Douala)": "Cameroonian",
    "Congolese": "Congolese",
    "Gambian": "Gambian",
    "Malian": "Malian",
    "sudanese": "Sudanese",
    "not specified": "Not Specified",
}

# Remove extra whitespace and map values
filtered_data["Nationality of the interviewee"] = (
    filtered_data["Nationality of the interviewee"].str.strip().map(nationality_mapping).fillna(filtered_data["Nationality of the interviewee"])
)

# Check the standardized values
print(filtered_data["Nationality of the interviewee"].value_counts())


Nationality of the interviewee
Somali       18
Sudanese     16
Nigerian     16
Chadian      13
Ethiopian    13
Libyan       12
Eritrean     12
Name: count, dtype: int64


In [17]:
'''
Sometimes, information like the "Year of birth" may not be recorded for every row associated 
with the same interviewee. Since a single interviewee can appear in multiple rows, we want 
to ensure that the year of birth is consistent across all these entries.

Process:
1. Group the dataset by 'Interviewee number'.
2. Within each group, use `ffill()` (forward fill) and `bfill()` (backward fill) to propagate 
   the year of birth values. This approach ensures that if any row for a given interviewee 
   lacks a recorded birth year, it inherits the value from another row for the same individual.

By doing this, we minimize missing values in "Year of birth" and maintain consistency 
in the dataset, facilitating more accurate analysis related to the interviewee's age.

After applying this transformation, we use `value_counts()` to verify the distribution 
of "Year of birth" and ensure the fills have been applied as intended.
'''



# Fill the 'Year of birth' for all rows corresponding to the same 'Interviewee number'.
filtered_data['Year of birth'] = filtered_data.groupby('Interviewee number')['Year of birth'].transform(lambda x: x.ffill().bfill())

# Verify the changes
filtered_data['Year of birth'].value_counts()


Year of birth
1996    9
1975    7
1974    5
1972    5
1999    5
2005    4
1979    4
1982    4
1990    4
2002    4
1993    4
2004    4
1983    3
1973    3
1978    3
1984    3
1971    3
1998    2
1992    2
2003    2
1970    2
2001    2
1981    2
1987    2
1991    2
1994    2
1986    2
1977    2
1980    1
1988    1
1989    1
1985    1
Name: count, dtype: int64

In [18]:
'''
Just as with 'Year of birth', some entries associated with the same interviewee might 
not record the "Date of the interview" for every row. To ensure data consistency, we 
again apply forward and backward filling on this column, grouped by 'Interviewee number'.

Process:
1. Group the data by 'Interviewee number'.
2. Use `transform()` with `ffill()` and `bfill()` to propagate the date of the interview 
   across all rows belonging to the same interviewee.
   
This ensures that each interviewee has a consistent interview date recorded for all 
of their associated rows, reducing missing values and improving the quality and 
coherence of the dataset. After this operation, we use `value_counts()` to verify 
the distribution of the "Date of the interview" values.
'''

filtered_data['Date of the interview'] = filtered_data.groupby('Interviewee number')['Date of the interview'].transform(lambda x: x.ffill().bfill())

# Verify the changes
filtered_data['Date of the interview'].value_counts()

Date of the interview
2024-08-02    2
2023-03-09    2
2024-06-23    2
2024-01-29    2
2024-03-28    2
             ..
2024-05-10    1
2023-05-04    1
2023-04-28    1
2023-10-12    1
2024-07-10    1
Name: count, Length: 94, dtype: int64

In [19]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Display the full content of each cell
pd.set_option('display.max_colwidth', None)

In [20]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Load the datase


'''
This code snippet computes the 'Age' of individuals from their recorded 'Year of birth' 
and 'Date of the interview'. In cases where the exact age isn't directly provided, 
the script attempts to calculate it based on the year of birth and the interview date. 
If either piece of information is missing or improperly formatted, the resulting age 
is marked as "not specified".

Key steps:
1. Standardize Columns:
   - Convert 'Date of the interview' and 'Year of birth' columns to strings and strip 
     whitespace to ensure consistent formatting.

2. Parse the Interview Date:
   - Attempt to parse the interview date using a known format ('%Y-%m-%d %H:%M:%S').
   - If the standard format fails, try other patterns (like a four-digit year or 
     a day-month-year format) to extract the interview year.
   - If parsing fails, return "not specified".

3. Determine Age:
   - If 'Year of birth' is a four-digit year, subtract it from the interview year 
     to calculate the age.
   - If 'Year of birth' contains a direct age or other number, extract it and use 
     that as the age.
   - If none of these conditions are met, return "not specified".

By applying these steps, the code transforms messy or incomplete data on birth years 
and interview dates into a more uniform 'Age' column. This processed data is then 
ready for further analysis or aggregation.
'''

# Function to calculate age or extract it from text
def calculate_age(row):
    # Extract Year of Birth
    yob = row['Year of birth']
    interview_date = row['Date of the interview']

    # Try to parse interview_date into a datetime object
    try:
        interview_year = datetime.strptime(interview_date, '%Y-%m-%d %H:%M:%S').year
    except:
        # Handle other formats
        if re.match(r'^\d{4}$', str(interview_date)):
            interview_year = int(interview_date)
        elif " " in str(interview_date) or "-" in str(interview_date):
            try:
                interview_year = datetime.strptime(interview_date, '%d %B %Y').year
            except:
                return "not specified"
        else:
            return "not specified"

    # Handle cases for year of birth
    if re.match(r'^\d{4}$', str(yob)):
        return interview_year - int(yob)
    elif isinstance(yob, str):
        age_match = re.search(r'(\d+)', yob)
        if age_match:
            return int(age_match.group(1))
        return "not specified"
    return "not specified"

# Standardize the 'Date of the interview' and 'Year of birth' columns for consistent formatting
filtered_data['Date of the interview'] = filtered_data['Date of the interview'].astype(str).str.strip()
filtered_data['Year of birth'] = filtered_data['Year of birth'].astype(str).str.strip()

# Apply the function to create the 'Age' column
filtered_data['Age'] = filtered_data.apply(calculate_age, axis=1)

# Save or display the resulting dataset

filtered_data[['Year of birth', 'Date of the interview', 'Age']].head(15)


,Year of birth,Date of the interview,Age
0,1971,2023-04-25,not specified
1,1987,2024-04-04,not specified
2,2004,2024-04-15,not specified
3,1993,2023-06-16,not specified
4,1983,2023-03-09,not specified
5,1977,2023-02-18,not specified
6,2002,2023-09-27,not specified
7,1975,2024-08-03,not specified
8,1974,2023-08-19,not specified
9,2004,2023-08-08,not specified


In [21]:
# Dropping the specified columns
filtered_data = filtered_data.drop(columns=['Year of birth', 'Date of the interview'])
filtered_data = filtered_data.dropna(how='all')
filtered_data["extracted_known_traffickers"].value_counts()

extracted_known_traffickers
unknown    100
Name: count, dtype: int64

raw to processed mapping of names based on the values above

In [22]:
'''
This code refines the mapping of known traffickers by performing a two-step normalization process:

1. **Raw-to-Processed Mapping:**
   The `raw_to_processed` dictionary maps complex strings, often containing multiple or 
   unclear names, to a cleaner, intermediate format. For instance, phrases like 
   "Not specified, Kidane" or "Eritrean And Abrham Weldemariam And Medhanie" 
   are simplified to standardized key terms like "Kidane" or "Medhanie and Abrham Weldemariam".
   
2. **Processed-to-Standardized Mapping:**
   The `standardized_mapping` dictionary then maps these processed keys 
   to fully standardized, canonical names as defined by a reference table or standard. 
   This ensures that all mentions of a trafficker, regardless of how they 
   were originally expressed, are consolidated under a single, known identifier.

Process:
- The `preprocess_and_map_trafficker` function:
  - Normalizes casing and spacing (e.g., `title()` for names, stripping whitespace).
  - Replaces commas with "and" to standardize delimiter usage.
  - Uses `raw_to_processed` to simplify complex or messy entries.
  - Uses `standardized_mapping` to map the simplified name to a final canonical version.
  
By applying this function to the 'extracted_known_traffickers' column, 
we ensure that all references to the same trafficker are standardized. 
This step is crucial for accurate analysis, such as frequency counts, 
cross-referencing, and establishing consistent entity recognition across the dataset.
'''


# Define the raw-to-processed mapping
raw_to_processed = {
    # Map sentences and complex phrases to standardized names
    "Not specified, Kidane": "Kidane",
    'Not Specified': "Unknown",
    "Eritrean And Abrham Weldemariam And Medhanie":"Abrham Weldemariam And Medhanie",
    'Wedi-Tesfay (Nickname)':'Unknown',
    "Mikele, chief oh the hotel), Mikele": "Mikele",
    "El amo, Abdel salam": "Abdel Salam",
    'Abdela Muhamad And Muhamad And Abdela':'Muhamad And Abdela',
    "The Short One And The Big One And Tadese": "Tadese",
    'Rasta (Smuggler) And Rasta':"Rasta",
    "Unknown , Unknown" : "Unknown",
    'Mamar Hagos And Hagos':'Hagos',
    "Tawilah And Tawila And Muhamad":"Tawila And Muhamad",
    "Eritrean And Medhanie Mered And Tin And Medhanie And Ermias":"Medhanie and Tin",
    "The Chief Of That Prison And I Think His Name Is Ibrahim.":'Ibrahim',
    "Unknown  And Unknown": "Unknown",
    "Not specified": "Unknown",
    "Not Reported": 'Unknown',
    "Not mentionned": "Unknown",
    "Unnamed": "Unknown",
    'Libyan Bandits And Bandits':'Unknown',
    "Not Mentionned": "Unknown",
    "Libyan bandits and Bandits": "Unknown",
    "Not reported": "Unknown",
    "Tin, Eritrean": "Tin",
    "Chadian soldiers, Kidane": "Kidane",
    "Wedi babu, Moussa": "Wedi Babu and Moussa",
    "Not specified, Mikele": "Mikele",
    "Muhamad wisky, Muhamad": "Muhamad Wisky",
    "Wedi isaac, Medhanie": "Wedi Isaac",
    "The short one and The big one and Tadese": "Unknown",
    "The chief of that prison and i think his name is ibrahim.": "Unknown",
    "Eritrean and Medhanie mered and Tin and Medhanie and Ermias": "Medhanie and Tin and Ermias",
    "Eritrean and Abrham weldemariam and Medhanie": "Medhanie and Abrham Weldemariam",
    "Tin and Tinaat": "Tin",
    "Aziz and Kidane and Abdel salam": "Aziz and Kidane and Abdel Salam",
    "Tawilah and Tawila and Muhamad": "Muhamad and Tawilah",
    "Rasta (smuggler) and Rasta": "Rasta",
    "Abdela muhamad and Muhamad and Abdela": "Abdela and Muhamad",
    "Wadi moussa and Moussa": "Moussa",
    "Mamar hagos and Hagos": "Hagos",
    "Sami and Samir": "Sami and Samir",
}

# Map raw processed names to the standardized names from the table
standardized_mapping = {
    # Mapping based on the provided table
    "Kidane": "Kidane",
    "Mikele": "Mikele",
    "Abdel Salam": "Abdel Salam",
    "Unknown": "Unknown",
    "Tin": "Tin",
    "Wedi Babu": "Wedi Babu",
    "Muhamad Wisky": "Muhamad Wisky",
    "Wedi Isaac": "Wedi Isaac",
    "Medhanie": "Medhanie",
    "Rasta": "Rasta",
    "Abdela": "Abdela",
    "Moussa": "Moussa",
    "Hagos": "Hagos",
    "Sami": "Sami",
}

# Apply the mapping function with preprocessing
def preprocess_and_map_trafficker(name):
    # Normalize spacing and casing
    name = name.strip().title()
    
    # Replace commas with "and"
    if "," in name:
        name = name.replace(",", " and")
    
    # Apply hardcoded replacements
    name = raw_to_processed.get(name, name)

    # Map the processed name to the standardized name
    return standardized_mapping.get(name, name)

# Apply the updated preprocessing and mapping function to the dataset
filtered_data['extracted_known_traffickers'] = filtered_data['extracted_known_traffickers'].apply(preprocess_and_map_trafficker)



handling exceptions

In [23]:
'''
This code further refines the preprocessing and standardization of trafficker names by handling 
some particularly difficult cases. While the initial raw-to-processed mapping handled 
many variants, certain complex entries still require more specific handling.

Key steps:
1. **Comma Replacement:**
   Commas in the original name strings are replaced with "and" to create a more uniform 
   delimiter between multiple entities.

2. **Hardcoded Replacements:**
   The `hardcoded_replacements` dictionary addresses known tricky cases where even 
   after initial processing, some strings remain ambiguous or overly complex.
   For example:
   - "Not specified and Kidane" -> "Kidane"
   - "Mikele and chief oh the hotel) and Mikele" -> "Mikele"
   - "Unknown and Unknown" -> "Unknown"

3. **Mapping to Standardized Names:**
   After applying these hardcoded corrections, the code calls `map_trafficker(name)` 
   (assumed to be defined elsewhere) to ensure that the final name string corresponds 
   to a known standardized name.

By repeatedly refining these mappings, we move from raw, inconsistent data towards 
a clean and reliable set of identifiers for each trafficker. This process ensures 
consistent data quality and supports meaningful analysis.

After applying the updated preprocessing and mapping, the code displays the first 
few rows of the 'extracted_known_traffickers' column to verify that the transformations 
have been applied correctly.
'''

# Further preprocessing for difficult cases
def preprocess_and_map_trafficker(name):
    # Replace commas with "and" for names with multiple entities
    if "," in name:
        name = name.replace(",", " and")
    
    # Handle specific known difficult cases
    hardcoded_replacements = {
        "Not specified and Kidane": "Kidane",
        "Mikele and chief oh the hotel) and Mikele": "Mikele",
        "El amo and Abdel salam": "Abdel Salam",
        "Unknown and Unknown": "Unknown",
        "Tin and Eritrean": "Tin",
        "Chadian soldiers and Kidane": "Kidane",
        "Wedi babu and Moussa": "Wedi Babu",
        "Not specified and Mikele": "Mikele",
        "Muhamad wisky and Muhamad": "Muhamad Wisky",
        "Wedi isaac and Medhanie": "Wedi Isaac",
    }

    # Apply hardcoded replacements
    name = hardcoded_replacements.get(name, name)

    # Map the processed name to the standardized name (assuming map_trafficker is defined previously)
    return name

# Apply the updated preprocessing and mapping function
filtered_data['extracted_known_traffickers'] = filtered_data['extracted_known_traffickers'].apply(preprocess_and_map_trafficker)



using standardized dict shared by the mentor to finalize the names 

In [24]:
'''
This code snippet provides an additional layer of data standardization for known trafficker 
and location names by mapping various alternative spellings and aliases to a single, 
consistent standardized name.

Process:
1. A `trafficker_mapping` dictionary defines the canonical names (keys) alongside 
   a list of alternative spellings or variations (values).
   
2. Using a dictionary comprehension, this mapping is "flattened" into `flat_mapping`, 
   where every alternative name directly maps to the standardized name. This simplifies 
   lookups and ensures that references to the same entity, regardless of how they're spelled, 
   converge to a single, canonical term.
   
3. The `map_trafficker` function uses `flat_mapping` to convert any alternative spelling 
   back to the standard name. If a name isn't found in the flat_mapping dictionary, 
   it remains unchanged, preserving data integrity.

4. Applying the `map_trafficker` function to the 'extracted_known_traffickers' column ensures 
   that all known variants of trafficker or location names are harmonized across the dataset.

After applying this mapping, the dataset contains a more uniform naming convention 
for traffickers and locations, facilitating clearer analysis, reporting, and pattern detection.
'''


# Create the dictionary for mapping trafficker names
trafficker_mapping = {
    # Standardized name: [alternative spellings]
    "Kidane": ["Kidani"],
    "Misrata": ["Misrâtah"],
    "Walid Isaac": ["Wali Isac"],
    "Abdel Salam": ["Abdoul Salam", "Abduselam", "Abdusalam", "Abdu Selam"],
    "Muhamad": ["Mohammed", "Mohamed"],
    "Enok": ["Enock"],
    "Abdela": ["Abdallah", "Abdalla", "Abdala"],
    "Moussa": ["Musa"],
    "Kashi": ["Keshi"],
    "Muhamad Wisky": ["Wisky"],
    "Unknown": ["Smuggler (unknown name)", "not reported", "Unknown", "Not Specified","unknown"],
    "Tripoli": ["Tarabulus"],
    "Kufrah": ["Koufrah", "Koufra", "Kufra", "Al-Kufrah"],
    "Triq al Matar": ["Shara al-Matar (also known as Tariq al-Matar)", "Triq al-Matar", "Tariq al Matar"],
    "Az Zawiyah": ["zaouia", "Az-Zawiya", "Zawiya"],
    "Bani Walid": ["Libya (Bani Walid)", "Bani Walid (Libya)", "Lybia (Bani-Walid)"],
    "Ash Shwayrif": ["Ash-Shwarif", "Ash-Shwayrif", "Ash Shwayrif (Libya)", "Ash-Swarif", "Shwayrif"],
    "Sabratha": ["Sabratah", "Zabratha", "Sabhrata"],
    "Brak Shati": ["Brak Shat", "Baraq Shati", "Braq Shati"],
    "Zuwarah": ["Suwara", "Zouara", "Zuwara", "Zwara"],
    "Ajdabiya": ["Adjdabiya"],
    "Ghiryan al Hamra": ["Ghiryan prison"],
    "Sabha": ["Sebha"],
    "Walid": ["Welid"],
    "Rufat": ["Refaat"],
    "Mikele": ["Michiele", "Miguel", "Michael"],
}

# Flatten the dictionary for direct mapping
flat_mapping = {alt: standard for standard, alts in trafficker_mapping.items() for alt in alts}
# Ensure any unmatched names are mapped to themselves
def map_trafficker(name):
    return flat_mapping.get(name, name)

# Apply the mapping to the 'extracted_known_traffickers' column in the dataset
filtered_data['extracted_known_traffickers'] = filtered_data['extracted_known_traffickers'].apply(map_trafficker)

# Display the first few rows to verify the mapping
filtered_data[['extracted_known_traffickers']].head()


,extracted_known_traffickers
0,Unknown
1,Unknown
2,Unknown
3,Unknown
4,Unknown


In [25]:
filtered_data.rename(columns={
    'abuse_type': 'abuse_type_experienced',
    'abuse_labels': 'abuse_labels_experienced'
}, inplace=True)

using standardized location names, based on the previous group's work from last year in order to maintain uniformity

In [26]:
# Skip loading external location reference file
# The location mapping dictionary in the next cells already contains
# all the standardized location names from previous year's work
print("Location standardization will use the comprehensive mapping dictionary below.")

Location standardization will use the comprehensive mapping dictionary below.


In [27]:
filtered_data.columns

Index(['Interviewee number', 'Interview ranking number',
       'Nationality of the interviewee', 'Gender interviewee',
       'Locations crossed (Place from)', 'Locations crossed (Place to)',
       'border crossed (undefined)', 'sexual_violence_experienced_binary',
       'sexual_violence_experienced_type', 'sexual_violence_witnessed_binary',
       'sexual_violence_witnessed_type', 'deaths_witnessed_binary',
       'deaths_witnessed_type', 'cleaned_payment_amount',
       'cleaned_payment_currency', 'extracted_known_traffickers',
       'abuse_type_experienced', 'abuse_labels_experienced', 'abuse_seen_type',
       'Age'],
      dtype='object')

In [28]:
filtered_data['Locations crossed (Place to)'].value_counts()

Locations crossed (Place to)
Tunisia                           6
Sudan                             3
Niger                             2
Ethiopia                          2
France                            2
Libya                             2
Chad                              2
Eritrea/Sudan/Tunisia/Egypt       1
Niger/Egypt/France                1
Sudan/Egypt/Italy                 1
Tunisia/Eritrea/Sudan             1
Sudan/Egypt/France                1
Eritrea/Chad/Sudan                1
Niger/France/Tunisia              1
Chad/Egypt/Sudan/France           1
Sudan/Chad                        1
Egypt/Libya/Niger                 1
Italy/Sudan/Ethiopia              1
Egypt/Tunisia                     1
Tunisia/Ethiopia/Niger            1
Tunisia/Italy/Sudan/Niger         1
Eritrea/Niger/Ethiopia            1
Eritrea/Sudan                     1
Egypt/Sudan/Chad                  1
Libya/Ethiopia                    1
Tunisia/Italy/Ethiopia/Sudan      1
Sudan/Tunisia                     1

Nationality mapping

In [29]:
'''
This code snippet refines the "Nationality of the interviewee" data by mapping specific nationalities 
and related terms to their corresponding standardized country names. In some cases, the data may 
include misspellings, alternate forms, or less common references to a nationality. By applying 
the `specific_nationality_mapping` dictionary, we ensure that these entries are converted 
into a consistent and recognized set of country names.

Process:
1. The `specific_nationality_mapping` dictionary associates various raw nationality labels (e.g., 
   "Eritrean", "Cameroonian", "Somalian", "senegalease") with their corrected or standardized country names 
   ("Eritrea", "Cameroon", "Somalia", "Senegal").
   
2. Using `map()` on the "Nationality of the interviewee" column, each original value is replaced 
   if it matches a key in the dictionary. If a value is not found in the dictionary, it remains 
   unchanged due to the `fillna()` fallback that reverts unmapped entries to their original form.

By applying this mapping, we achieve a cleaner and more uniform representation of 
interviewee nationalities, simplifying subsequent analyses, comparisons, and reporting.
'''


# Define the corrected mapping for specific values
specific_nationality_mapping = {
    
    'Eritrean': 'Eritrea',
    'Cameroonian': 'Cameroon',
    'Senegalese': 'Senegal',
    'Bissau-Guinean': 'Guinea-Bissau',
    'Not Specified': 'Unknown',
    'Nigerian': 'Nigeria',
    'Sudanese': 'Sudan',
    'Gambian': 'Gambia',
    'Guinean': 'Guinea',
    'Congolese': 'Democratic Republic of the Congo',
    'Tunisian': 'Tunisia',
    'Somali': 'Somalia',
    'Malian': 'Mali',
    'Ivorian': 'Ivory Coast',
    'senegalease': 'Senegal',  # Correct spelling
    'Somalian': 'Somalia',  
    "Khartoum, Sudan": "Sudan",
    "Somalia": "Somalia",
    "Malta": "Malta",
    "Guinea": "Guinea",
    "France": "France",
    "Congo": "Republic of the Congo",
    "Benin": "Benin",
    "Netherlands": "Netherlands",
    "Uganda": "Uganda",
    "Aljmail": "Libya",
    "Chad": "Chad",
    "Gambia": "Gambia",
    "Spain": "Spain",
    "Yemen": "Yemen",
    "Republic of the Congo": "Republic of the Congo",
    "Dahman": "Libya",
    "Zuwarah": "Libya",
    "Central African Republic": "Central African Republic",
    "Ghana": "Ghana",
    "Togo": "Togo",
    " Nigeria": "Nigeria",  
}

# Apply the mapping to update the column with corrected values
filtered_data['Nationality of the interviewee'] = (
    filtered_data['Nationality of the interviewee']
    .map(specific_nationality_mapping)
    .fillna(filtered_data['Nationality of the interviewee'])  # Retain unmapped values
)




Location mapping, can you used to standardize and column which has location info, might contain redundant info because it is created using local llm, but if wont affect the final dataset

In [30]:
'''
This code snippet standardizes the names of locations in the "Locations crossed (Place to)" 
column to ensure data consistency. Given the complexity and variability of place names— 
including typos, alternative spellings, unnecessary whitespace, and different language 
representations—this mapping helps unify these references into a coherent set of standardized 
geographical labels.

Process:
1. Define a comprehensive `location_mapping` dictionary that associates various raw strings 
   (including misspellings, extra whitespace, country/city pairs, and different language forms) 
   with a single standardized location name.
   
2. Strip leading and trailing whitespace from the "Locations crossed (Place to)" column 
   to minimize matching errors due to formatting inconsistencies.

3. Use the `.map()` method with `location_mapping` to replace each entry with its 
   standardized counterpart. If a value doesn't appear in the dictionary, it remains unchanged.

By applying this mapping, the dataset gains a more uniform geography field, supporting 
better analytics, reporting, and data-driven insights related to migration routes 
and border crossings.
'''


# Define the mapping dictionary
location_mapping = {
    "Burkina Faso": "Burkina Faso",
    "Ghana": "Ghana",
    "Togo": "Togo",
    "Malta": "Malta",
    "Senegal": "Senegal",
    "Angola": "Angola",
    "Netherlands (plane)": "Netherlands",
    "Republic of the Congo": "Republic of the Congo",
    "Central African Republic": "Central African Republic",
    "Cameroon": "Cameroon",
    "Saudi Arabia": "Saudi Arabia",
    "Eritrea": "Eritrea",
    "Israel": "Israel",
    " Niger": "Niger",
    
    # Fix typos and alternate representations
    " Burkina": "Burkina Faso",
    " Angola": "Angola",
    " Niger": "Niger",
    " Netherlands (plane)": "Netherlands",
    # Correct direct matches
        "Malta": "Malta",
    "Republic of the Congo": "Republic of the Congo",
    "Ghana": "Ghana",
    "Burkina": "Burkina Faso",
    "Angola": "Angola",
    "Israel": "Israel",
    "Central africa": "Central African Republic",
    "Cameroon": "Cameroon",
    "Saudi Arabia": "Saudi Arabia",
    "Netherlands (plane)": "Netherlands",
    "Italy": "Italy",
    "Eritrea": "Eritrea",
    "Modjroun (Libya)": "Libya",
    "Qatrun (Libya)": "Libya",
    "Niger (by plane aranged by UNHCR)": "Niger",
    "Niger": "Niger",
    
    # Fix typos and alternate representations
    " Burkina": "Burkina Faso",
    " Niger": "Niger",
    " Modjroun (Libya)": "Libya",
    " Qatrun (Libya)": "Libya",
    "Eritrea": "Eritrea",
    "Ethiopia": "Ethiopia",
    "Sudan": "Sudan",
    "Libya": "Libya",
    "Niger": "Niger",
    "Nigeria": "Nigeria",
    "Algeria": "Algeria",
    "Cameroon": "Cameroon",
    "Somalia": "Somalia",
    "Mali": "Mali",
    "Senegal": "Senegal",
    "Egypt": "Egypt",
    "Italy": "Italy",
    "France": "France",
    "Germany": "Germany",
    "Switzerland": "Switzerland",
    "Spain": "Spain",
    "Ivory Coast": "Ivory Coast",
    "Guinea": "Guinea",
    "Ghana": "Ghana",
    "Morocco": "Morocco",
    "Chad": "Chad",
    "Tunisia": "Tunisia",
    "Burkina Faso": "Burkina Faso",
    "Saudi Arabia": "Saudi Arabia",
    "Yemen": "Yemen",
    "Uganda": "Uganda",
    "Israel": "Israel",
    "Benin": "Benin",
    "Zambia": "Zambia",
    "Mozambique": "Mozambique",
    "Guinea-Bissau": "Guinea-Bissau",
    "Khartoum, Sudan": "Sudan",
    "Somalia": "Somalia",
    "Malta": "Malta",
    "Guinea": "Guinea",
    "France": "France",
    "Congo": "Republic of the Congo",
    "Benin": "Benin",
    "Netherlands": "Netherlands",
    "Uganda": "Uganda",
    "Aljmail": "Libya",
    "Chad": "Chad",
    "Gambia": "Gambia",
    "Spain": "Spain",
    "Yemen": "Yemen",
    "Republic of the Congo": "Republic of the Congo",
    "Dahman": "Libya",
    "Zuwarah": "Libya",
    "Central African Republic": "Central African Republic",
    "Ghana": "Ghana",
    "Togo": "Togo",
    " Nigeria": "Nigeria",
    
    # Fix typos and alternate representations
    "eritrea": "Eritrea",
    "libya": "Libya",
    "niger": "Niger",
    "Lybia": "Libya",
    "Libya ": "Libya",
    "Tunisia ": "Tunisia",
    "Cameroon ": "Cameroon",
    "Libya, Kufrah": "Libya",
    "Libya, Ash Shwayrif": "Libya",
    "Libya, Sabratha": "Libya",
    "Ghat (Libya)": "Libya",
    "Bani Walid, Libya": "Libya",
    "Tarabulus, Libya": "Libya",
    "Ghadames (Libya)": "Libya",
    "Libya, Ghiryan al Hamra": "Libya",
    "Libya, Zintan": "Libya",
    "Libya, Tripoli": "Libya",
    "Libya, Morzouk": "Libya",
    "Libya, Gatrone": "Libya",
    "Libya, Sabha": "Libya",
    "Libya, Niger": "Libya",
    "Bani Walid (Libya)": "Libya",
    "Ash Shwayrif, Libya": "Libya",
    "Sabratha, Libya": "Libya",
    "Tariq al Matar, Libya": "Libya",
    "Tajura, Libya": "Libya",
    "Garaboli (Libya)": "Libya",
    "Tripoli (Libya)": "Libya",
    "Tripoli (Triq al-matar)": "Libya",
    "Zuwarah, Libya": "Libya",
    "Place in the mountains (Libya)": "Libya",
    "House (Libya)": "Libya",
    "Police station (Libya)": "Libya",
    "Place of Red Cross, Sabratha (Libya)": "Libya",
    "Ghiryan al Hamra prison": "Libya",
    "Triq al Matar prison": "Libya",
    
    # Locations and borders
    "Eritrea to Ethiopia": "Eritrea",
    "Ethiopia to Sudan": "Ethiopia",
    "From Ethiopia to Sudan": "Ethiopia",
    "Ethiopia to Libya": "Ethiopia",
    "Sudan to Libya": "Sudan",
    "From Sudan to Libya": "Sudan",
    "From Libya to Niger": "Libya",
    "Sudan to Egypt": "Sudan",
    "Libya to Tunisia": "Libya",
    "From Libya to Tunisia": "Libya",
    "Niger to Libya": "Niger",
    "Eritrea to Libya to Niger": "Eritrea",
    "Eritrea to Sudan": "Eritrea",
    "From Eritrea to Sudan": "Eritrea",
    "Ethiopia, Dembe Gedamu": "Ethiopia",
    "Border between Ethiopia and Sudan": "Ethiopia",
    "Border between Sudan and Libya": "Sudan",
    "From Algeria to Libya": "Algeria",
    "Sudan to Libya as kidnapped": "Sudan",
    
    # Complex journey descriptions
    "Journey from Ethiopia to Sudan": "Ethiopia",
    "Journey through the Sahara towards Libya": "Libya",
    "Before I came to Libya, I passed through the Sahara Desert with a lot of Somali and Eritreans. But the majority of them were Eritreans people": "Eritrea",
    "Guinea to Mali": "Guinea",
    "Guinea to Mali to Algeria (summary)": "Guinea",
    "From libya to sea to Tunisia": "Libya",
    "This line focuses on things mentioned about Libya in general": "Libya",
    
    # Unknown and unspecified entries
    "Unknown": "Unknown",
    "Not clear, the interview starts when interviewee already was in Libya": "Libya",
    "not reported": "Unknown",
    "... Sudan": "Sudan",
    "border quoted": "Unknown",
    "borders quoted": "Unknown",
    "END OF SUMMARY": "Unknown",
        "Eritrea": "Eritrea",
    "Ethiopia": "Ethiopia",
    "Sudan": "Sudan",
    "Libya": "Libya",
    "Niger": "Niger",
    "Nigeria": "Nigeria",
    "Algeria": "Algeria",
    "Cameroon": "Cameroon",
    "Somalia": "Somalia",
    "Mali": "Mali",
    "Senegal": "Senegal",
    "Egypt": "Egypt",
    "Italy": "Italy",
    "France": "France",
    "Germany": "Germany",
    "Switzerland": "Switzerland",
    "Spain": "Spain",
    "Ivory Coast": "Ivory Coast",
    "Guinea": "Guinea",
    "Ghana": "Ghana",
    "Morocco": "Morocco",
    "Chad": "Chad",
    "Tunisia": "Tunisia",
    "Burkina Faso": "Burkina Faso",
    "Saudi Arabia": "Saudi Arabia",
    "Yemen": "Yemen",
    "Uganda": "Uganda",
    "Israel": "Israel",
    "Benin": "Benin",
    "Zambia": "Zambia",
    "Mozambique": "Mozambique",
    "Guinea-Bissau": "Guinea-Bissau",
    
    # Fix typos and alternate representations
    "Eriteria": "Eritrea",
    "eritrea": "Eritrea",
    "libya": "Libya",
    "LIbya": "Libya",
    "Lybia": "Libya",
    "Sudan, Khartoum": "Sudan",
    "sudan": "Sudan",
    "Niger, LIbya": "Niger",
    "Niger, Libya": "Niger",
    "NIger, Libya": "Niger",
    "tunisia": "Tunisia",
    "Tunisia ": "Tunisia",
    "tunisia but talikng about libya": "Tunisia",
    "Camerron": "Cameroon",
    "Cameroon ": "Cameroon",
    "Gambia": "Gambia",
    "Senegal ": "Senegal",
    "senengal": "Senegal",
    "Mediterranean sea": "Mediterranean Sea",
    "mediterranean sea": "Mediterranean Sea",
    "Algiers, Algeria": "Algeria",
    "Alger (Algeria)": "Algeria",
    "Tamanrasset (Algeria)": "Algeria",
    "Tamanrasset, Algeria": "Algeria",
    "Adrar ( Algeria)": "Algeria",
    "Oujda, Morocco": "Morocco",
    "Oujda (Marocco)": "Morocco",
    "Marocco": "Morocco",
    
    # Normalize descriptions of borders and routes
    "from Libya to Tunisia": "Libya",
    "from Algeria to Libya": "Algeria",
    "From Ethiopia to Sudan, from Sudan to Libya.": "Ethiopia",
    "Sudan to Libya.": "Sudan",
    "Sudan to Libya": "Sudan",
    "Niger, Agadez": "Niger",
    "Niger(Agdez)": "Niger",
    "Agadez (Niger)": "Niger",
    "Agadez (Rimbo station)": "Niger",
    "Niger (Niamey)": "Niger",
    "Assamakka, Niger": "Niger",
    "Arlit, Niger": "Niger",
    "Agadez, Niger": "Niger",
    "Eritrea (Sawa) - Sudan (Kassala)": "Eritrea",
    "Eritrea to Sudan": "Eritrea",
    "from Eritrea to Sudan": "Eritrea",
    "from Eritrea to Ethiopia": "Eritrea",
    "Eritrea to Ethiopia": "Eritrea",
    "Ethiopia - South Sudan (Juba)": "Ethiopia",
    "Ethiopia to Sudan": "Ethiopia",
    "from Ethiopia to Sudan": "Ethiopia",
    "Sudan - Egypt (Aswan": "Sudan",
    "from Sudan to Libya": "Sudan",
    "from Sudan to south sudan": "Sudan",
    "South Sudan (Juba) - Sudan (Khartoum)": "South Sudan",
    "South Sudan": "South Sudan",
    "Juba, South Sudan": "South Sudan",
    "from Sudan to South Sudan": "Sudan",
    
    # Specific places in Libya
    "Bani Walid, Libya": "Libya",
    "Libya, Bahi": "Libya",
    "Libya, Niger quoted": "Libya",
    "Libya, Sabha": "Libya",
    "Libya, Triq al Matar": "Libya",
    "Libya, Az Zawiyah": "Libya",
    "Tripoli, Libya": "Libya",
    "Tarabulus (Tripoli), Libya": "Libya",
    "Misrata": "Libya",
    "Zuwarah": "Libya",
    "Zuwarah, Libya": "Libya",
    "Sabha": "Libya",
    "Sabha, Libya": "Libya",
    "Umm al Aranib, Libya": "Libya",
    "Ash Shwayrif, Libya": "Libya",
    "Sabratha, Libya": "Libya",
    "Libya port": "Libya",
    "Unknown place in Libya": "Libya",
    
    
    # Journey descriptions
    "Journey through the Sahara towards Libya": "Libya",
    "From Libya to Niger": "Libya",
    "Journey from Ethiopia to Sudan": "Ethiopia",
    
    # Unknown or unclassifiable
    "Unknown": "Unknown",
    "not reported": "Unknown",
    "Unknown place": "Unknown",
        "from Mali to Libya": "Mali",
    "Somalia": "Somalia",
    "Spain": "Spain",
    "Debdeb (Algeria)": "Algeria",
    "Chad": "Chad",
    "Yemen": "Yemen",
    "niger in december 2018": "Niger",
    "Burkina": "Burkina Faso",
    "eritrea to sudan": "Eritrea",
    "ethiopia to sudan": "Ethiopia",
    "libya - Tunisia": "Libya",
    "Nigeria, Ivory Coast, Burkina Faso": "Nigeria",
    "Niger, Algeria, Libya": "Niger",
    "libya and then tunisia": "Libya",
    " Togo": "Togo",
    "they are in Tunisia but they are talking about libya": "Tunisia",
    "Ghana": "Ghana",
    "Sudan": "Sudan",
    "Benin": "Benin",
    " Nigeria": "Nigeria",
    "Algeria, NIgeria": "Algeria",
    "Mali (Gao)": "Mali",
    "Malta and then returned to Tunisia": "Malta",
    "Bordj Badji Mokhtar (Algeria)": "Algeria",
    "Ivory Coast, Abidjan": "Ivory Coast",
    "NIger": "Niger",
    "Nigeria to Libya. sometimes they have to pass through Chad": "Nigeria",
    "al-Khalīl (Algeria)": "Algeria",
    "Niger and then he came back to Niger": "Niger",
    "Niger, Dirkou": "Niger",
    "Ghadames (Libya": "Libya",
    "Kumba, Cameroon": "Cameroon",
    "Fron Eritrea to Ethiopia": "Eritrea",
    "Burkina Faso, Ouagadougou": "Burkina Faso",
    "Mali, Bamako": "Mali",
    "no more Mali-niger route cause of problems in mali": "Mali",
    "Somalia and Ethiopia": "Somalia",
    "Telem5": "Unknown",
    "Mali (Bamako)": "Mali",
    "Nantes, France": "France",
    "libya, talking about prison": "Libya",
    "Benghazi, Libya": "Libya",
    "Triq al Matar": "Libya",
    "Sabha": "Libya",
    "Agadez": "Niger",
    "Tesen, Eritrea": "Eritrea",
    "Hajer, Sudan": "Sudan",
    "Shagarab, Sudan": "Sudan",
    "Sinai, Egypt": "Egypt",
    "Saudi Arabia": "Saudi Arabia",
    "Guinea-Bissau": "Guinea-Bissau",
    "Central African Republic": "Central African Republic",
    "Republic of the Congo": "Republic of the Congo",
    "Congo-Kinshasa": "Democratic Republic of the Congo",
    "Mozambique": "Mozambique",
    "Zambia": "Zambia",
    "Milan, Italy": "Italy",
    "Kalil between Mali and Algeria": "Mali",
    "Switzerland": "Switzerland",
    "Germany": "Germany",
    "France": "France",
    "Calais, France": "France",
    "desert to Algeria": "Algeria",
    "Congo Brazaville": "Republic of the Congo",
    "endabaguna, Ethiopia": "Ethiopia",
    "Adi Harush, Ethiopia": "Ethiopia",
    "Addis Abeba, Ethiopia": "Ethiopia",
    "Sikka, Libya": "Libya",
    "Garabulli [Casterverde], Libya": "Libya",
    "Guinea": "Guinea",
    "Mali (seven border stations between Mali and Algeria)": "Mali",
    "algerian desert": "Algeria",
    "Egypte": "Egypt",
        "Assamaka (Niger)": "Niger",
    "In Guezzam (Algeria)": "Algeria",
    "Tripoli": "Libya",
    "Sabha": "Libya",
    "Niamey (NIger)": "Niger",
    "Tamba (Senegal)": "Senegal",
    "Sinai": "Egypt",
    "Sudan, Hajer": "Sudan",
    "Israel": "Israel",
    "libya to tunisia": "Libya",
    "from south sudan to sudan": "South Sudan",
    "from sudan to south sudan": "Sudan",
    "from eritrea to ethiopia": "Eritrea",
    "Libya (Sabha)": "Libya",
    "Modjroun": "Libya",
    "Qatrun": "Libya",
    "Algeria - Libya": "Algeria",
    "Libya (Bani Walid) - Algeria (Az Zawiyah)": "Libya",
    "Sudan (Khartoum) - Libya (Bani Walid)": "Sudan",
    "Egypt - Ethiopia": "Egypt",
    "European border": "Unknown",
    "Border of Israel": "Israel",
    "Isreal, Ashkelon": "Israel",
    "Omdurman, Sudan": "Sudan",
    "Place of the smugglers; close to the border between Ethiopia and Sudan": "Ethiopia",
    "Golij, Gashbarka, Eritrea": "Eritrea",
    "Asmara, Eritrea": "Eritrea",
    "Gahtelay, Eritrea": "Eritrea",
    "Mietr, Eritrea": "Eritrea",
    "Ivory coast": "Ivory Coast",
    "from Libya to Niger": "Libya",
    "Hitsats, Ethiopia": "Ethiopia",
    "Zambia": "Zambia",
    "Mozambique": "Mozambique",
    "Congo-Brazzaville\n": "Republic of the Congo",
    "Central Africa": "Central African Republic",
    "Guinea-Bissau": "Guinea-Bissau",
    "Mai Aini, Ethiopia": "Ethiopia",
    "Uganda": "Uganda",
    "Uganda to Sudan": "Uganda",
    "Israel to Uganda": "Israel",
    "Egypt to Israel": "Egypt",
    "entering Sudan": "Sudan",
    "Niamey, Niger": "Niger",
    "Libya, Bani Walid": "Libya",
    "Sudan, Shagarab": "Sudan",
    "Sudan, Wad Sherife": "Sudan",
    "Eritrea, Forto Sawa": "Eritrea",
    "Saudi Arabia": "Saudi Arabia",
    "sudan to Libya": "Sudan",
    "Kampala, Uganda": "Uganda",
    "Ethioipia": "Ethiopia", 
      "Libya": "Libya",
    "Sudan": "Sudan",
    "Tunisia": "Tunisia",
    "Ethiopia": "Ethiopia",
    "Niger": "Niger",
    "Nigeria": "Nigeria",
    "Mediterranean Sea": "Mediterranean Sea",
    "Yemen": "Yemen",
    "Egypt": "Egypt",
    "Chad": "Chad",
    "Mali": "Mali",
    "Malta": "Malta",
    "Ivory Coast": "Ivory Coast",
    "Mozambique": "Mozambique",
    "Burkina Faso": "Burkina Faso",
    "Senegal": "Senegal",
    "Togo": "Togo",
    "Benin": "Benin",
    "Angola": "Angola",
    "Saudi Arabia": "Saudi Arabia",
    "Central Africa": "Central African Republic",
    "Cameroon": "Cameroon",
    "Israel": "Israel",
    "Italy": "Italy",
    "Netherlands (plane)": "Netherlands",
    "Niger (Agadez)": "Niger",
    "Burkina Faso": "Burkina Faso",
    "Algeria": "Algeria",
    "Libya  (Ras Jdir)": "Libya",
    "Qatrun (Libya)": "Libya",
    "Modjroun (Libya)": "Libya",
    "Sabha (Libya)": "Libya",
    "unknown/Libya": "Libya",
    "\nCongo-Brazzaville": "Republic of the Congo",
    " Ghana": "Ghana",
    " Ethiopia": "Ethiopia",
    " Niger": "Niger",
    " Burkina": "Burkina Faso",
    " Algeria": "Algeria",
    " \nNigeria": "Nigeria",
    "Sinai)": "Egypt",
    "Sinai(?)": "Egypt",
    "Ethiopia (deported)": "Ethiopia",
    "Tunisia (via the sea)": "Tunisia",
    "Burkina/ \\nNiger (Rimbo station, \nAgadez \nvia SONEF Transport service)": "Niger",
    "Libya (he does not mention how he entered Libya, through which city": "Libya",
    
    # Fix typos and alternate representations
    " Sudan": "Sudan",
    " Libya": "Libya",
    " Niger": "Niger",
    " Ivory Coast": "Ivory Coast",
    " Benin": "Benin", # Correcting typo
}

# Apply mapping function
# Strip leading and trailing whitespace from the 'Nationality of the interviewee' column
filtered_data['Locations crossed (Place to)'] = (
    filtered_data['Locations crossed (Place to)']
    .str.strip()  # Remove whitespace from start and end
    .map(location_mapping)  # Apply the mapping
    .fillna(filtered_data['Locations crossed (Place to)'])  # Retain unmapped values
)



In [31]:
# Apply the same location mapping to "Locations crossed (Place from)" column
filtered_data['Locations crossed (Place from)'] = (
    filtered_data['Locations crossed (Place from)']
    .str.strip()  # Remove whitespace from start and end
    .map(location_mapping)  # Apply the mapping
    .fillna(filtered_data['Locations crossed (Place from)'])  # Retain unmapped values
)

print("Location standardization complete for both 'Place from' and 'Place to' columns.")

Location standardization complete for both 'Place from' and 'Place to' columns.


In [32]:
import json

'''
This code snippet converts the processed DataFrame rows into a structured JSON format. 
By iterating over each row of the `filtered_data` DataFrame and extracting relevant 
fields, it creates a nested JSON structure that includes victim information, crimes, 
border crossings, and extortion amounts.

Key steps:

1. **Defining the JSON Structure (`create_json` function)**:
   - `victim_id`: Unique identifier of the victim (e.g., "Interviewee number").
   - `age`, `gender`, `nationality`: Personal details about the victim.
   - `trafficker_name`: Name or identifier of the known trafficker.
   - `crimes`: A list of crime-related dictionaries containing details of sexual violence 
     experienced or witnessed, deaths witnessed, and abuses (both experienced and seen).
   - `borders_crossed`: A list of dictionaries representing the places from and to 
     which the victim crossed borders.
   - `money_extorted_amount`, `money_extorted_currency`: Financial extortion details, 
     standardized and extracted from payment data.

2. **Generating the JSON List**:
   - The code applies the `create_json` function to each row of the `filtered_data` DataFrame, 
     building a list (`json_data`) of JSON-compatible Python dictionaries.

3. **Conversion and Saving**:
   - The `json_data` list is then converted into a formatted JSON string (`json_output`) 
     with `json.dumps(json_data, indent=4)` for better readability.
   - Finally, the code writes this JSON output to a file named 'victims.json'.

This structured JSON makes it easier to integrate with other systems, conduct analytics, 
or store the processed information for future reference.
'''


# Define the JSON structure based on the provided template
def create_json(row):
    return {
        "victim_id": row["Interviewee number"],
        "age": row["Age"],
        "gender": row["Gender interviewee"],
        "nationality": row["Nationality of the interviewee"],
        "trafficker_name": row["extracted_known_traffickers"],
        "crimes": [
            {
                "sexual_violence_experienced_binary": row["sexual_violence_experienced_binary"],
                "sexual_violence_experienced_type": row["sexual_violence_experienced_type"]
            },
            {
                "sexual_violence_witnessed_binary": row["sexual_violence_witnessed_binary"],
                "sexual_violence_witnessed_type": row["sexual_violence_witnessed_type"]
            },
            {
                "deaths_witnessed_binary": row["deaths_witnessed_binary"],
                "deaths_witnessed_type": row["deaths_witnessed_type"]
            },
            {
                "abuse_type_experienced": row["abuse_type_experienced"],
                "abuse_labels_experienced": row["abuse_labels_experienced"],
                "abuse_seen_type": row["abuse_seen_type"]
            }
        ],
        "borders_crossed": [
            {"border": row["Locations crossed (Place from)"]},
            {"border": row["Locations crossed (Place to)"]}
        ],
        "money_extorted_amount": row["cleaned_payment_amount"],
        "money_extorted_currency": row["cleaned_payment_currency"]
    }

# Apply the function to each row of the dataframe
json_data = [create_json(row) for _, row in filtered_data.iterrows()]

# Convert the list of JSON objects to a JSON string for visualization or saving
json_output = json.dumps(json_data, indent=4)

# Save to a file or display
output_path = 'victims.json'
with open(output_path, 'w') as f:
    f.write(json_output)


